```c#
netw intrusion detection sys    |     |
                                \\_V_//
                                \/=|=\/
                                 [=v=]
                               __\___/_____
                              /..[  _____  ]
                             /_  [ [  M /] ]
                            /../.[ [ M /@] ]
                           <-->[_[ [M /@/] ]
                          /../ [.[ [ /@/ ] ]
     _________________]\ /__/  [_[ [/@/ C] ]
    <_________________>>0---]  [=\ \@/ C / /
       ___      ___   ]/000o   /__\ \ C / /
          \    /              /....\ \_/ /
       ....\||/....           [___/=\___/
      .    .  .    .          [...] [...]
     .      ..      .         [___/ \___]
     .    0 .. 0    .         <---> <--->
  /\/\.    .  .    ./\/\      [..]   [..]
 / / / .../|  |\... \ \ \    _[__]   [__]_
/ / /       \/       \ \ \  [____>   <____]
```

~ `snuff.ipynb` by kriston

### install scapy
used for creating, sending, capturing, and analyzing network packets

In [10]:
!pip install scapy

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


### more setup
1. run the nids vm
2. open attacker and client ui
3. wireshark -> vmnet1 / vmnet3 (mac)

### level 1.1 - intro
create the basic skeleton and implement the first detection

In [ ]:
from scapy.all import *
from datetime import datetime, timedelta
import platform


# Threat intelligence
# ADD YOUR C2 SERVER IPS HERE
MALICIOUS_IPS = ["4.2.2.2"]

# dict to track alerts
alert_history = {}

# set a timeout to ignore repeated alerts within 10s
REPEATED_ALERT_TIMEOUT = timedelta(seconds=10)


def get_interface():
    """
    checks for the OS and returns the correct network interface
    """
    print("Checking for OS...")
    if platform.system() == "Windows":
        print("OS is Windows -> VMware Network Adapter VMnet1")
        return "VMware Network Adapter VMnet1"
    elif platform.system() == "Darwin":  # macOS is identified as 'Darwin'
        print("OS is Mac -> vmenet3")
        return "vmenet3"
    else:
        raise RuntimeError("Only supports windows/mac")


def alert(msg):
    """
    alerts function to check if the alert has been logged before within the timeout
    """
    # ignore the alert if it has already been logged before timeout
    if (msg in alert_history) and (
        alert_history[msg] + REPEATED_ALERT_TIMEOUT > datetime.now()
    ):
        return
    # log the alert's msg and timestamp
    alert_history[msg] = datetime.now()
    print(f"*ALERT* {msg}")


def detect_malicious_ip(packet):
    """
    Detection 1.1
    Detect communication with known malicious IPs.
    """
    # YOUR CODE BELOW
    # Check source IP
    if packet[IP].src in MALICIOUS_IPS:
        alert(f"Malicious IP {MALICIOUS_IPS} communicating with destination: {packet[IP].dst}")
    
    # Check destination IP
    if packet[IP].dst in MALICIOUS_IPS:
        alert(f"Malicious IP {MALICIOUS_IPS} communicating with source: {packet[IP].src}")


def detect_malicious_traffic(packet):
    """
    function to detect malicious network activity
    """
    try:
        # check if the packet has IP layer
        if packet.haslayer(IP):
            detect_malicious_ip(packet)
    # print any exceptions and continue (to prevent crashing)
    except Exception as e:
        print(f"Error: {e}")


def main():
    print("Starting NIDS!")
    network_interface = get_interface()
    print(f"Sniffing on interface: {network_interface}")
    sniff(filter="ip", prn=detect_malicious_traffic, iface=network_interface)


if __name__ == "__main__":
    main()


Starting NIDS!
Checking for OS...
OS is Mac -> vmenet3
Sniffing on interface: vmenet3
*ALERT* Malicious IP ['4.2.2.2'] communicating with source: 10.0.0.13
*ALERT* Malicious IP ['4.2.2.2'] communicating with destination: 10.0.0.13


### level 1.2 - DNS resolving of a known malware domain
prev our IOC (Indicator of compromise) was a known malicious IP, this time, the IOC is a known malicious domain name

In [ ]:
from scapy.all import *
from datetime import datetime, timedelta
import platform


# Threat intelligence
# ADD YOUR C2 SERVER IPS HERE
MALICIOUS_IPS = ["4.2.2.2"]

# dict to track alerts
alert_history = {}

# set a timeout to ignore repeated alerts within 10s
REPEATED_ALERT_TIMEOUT = timedelta(seconds=10)


def get_interface():
    """
    checks for the OS and returns the correct network interface
    """
    print("Checking for OS...")
    if platform.system() == "Windows":
        print("OS is Windows -> VMware Network Adapter VMnet1")
        return "VMware Network Adapter VMnet1"
    elif platform.system() == "Darwin":  # macOS is identified as 'Darwin'
        print("OS is Mac -> vmenet3")
        return "vmenet3"
    else:
        raise RuntimeError("Only supports windows/mac")


def alert(msg):
    """
    alerts function to check if the alert has been logged before within the timeout
    """
    # ignore the alert if it has already been logged before timeout
    if (msg in alert_history) and (
        alert_history[msg] + REPEATED_ALERT_TIMEOUT > datetime.now()
    ):
        return
    # log the alert's msg and timestamp
    alert_history[msg] = datetime.now()
    print(f"*ALERT* {msg}")


def detect_malicious_ip(packet):
    """
    Detection 1.1
    Detect communication with known malicious IPs.
    """
    # YOUR CODE BELOW
    # Check source IP
    if packet[IP].src in MALICIOUS_IPS:
        alert(f"Malicious IP {MALICIOUS_IPS} communicating with destination: {packet[IP].dst}")
    
    # Check destination IP
    if packet[IP].dst in MALICIOUS_IPS:
        alert(f"Malicious IP {MALICIOUS_IPS} communicating with source: {packet[IP].src}")


def detectmaliciousdns(packet):
    """
    Detection 1.2
    Detect DNS requests of known malicious domains
    """
    # Check if the packet has DNS layer
    if packet.haslayer(DNS) and packet[DNS].qr == 0:  # 0 means query
        domain = packet[DNS].qd.qname.decode('utf-8')
        if domain in MALICIOUS_IPS:  # Assuming MALICIOUS_IPS contains domains as well
            alert(f"Malicious DNS request for domain: {domain}")


def detect_malicious_traffic(packet):
    """
    function to detect malicious network activity
    """
    try:
        # check if the packet has IP layer
        if packet.haslayer(IP):
            detect_malicious_ip(packet)

    except Exception as e:
        print(f"Error: {e}")


def main():
    print("Starting NIDS!")
    network_interface = get_interface()
    print(f"Sniffing on interface: {network_interface}")
    sniff(filter="ip", prn=detect_malicious_traffic, iface=network_interface)


if __name__ == "__main__":
    main()


### 1.3 malware signature in HTTP
implement  DPI  (Deep-Packet-Inspection)  and  detect  malware downloads over HTTP

In [20]:
from scapy.all import *
from datetime import datetime, timedelta
import platform
import hashlib


# Threat intelligence
MALICIOUS_IPS = ["4.2.2.2"]
MALICIOUS_DOMAINS = ["virus.com"]
MALICIOUS_HASHES = {
    "b3f067e63fff8e171ee26bcde6a6010737c8b22c": "LokiBot",
    "73480f2548244fb6a3e9db83a4e74082dd2fa500": "LokiBot",
    "efbd4555c4b881d77d28f659289373a813e79650": "TeslaCrypt", 
    "13427e27f405ea2c818d4f55745cd9fb9e336134": "TeslaCrypt"
}

# Dict to track TCP streams
tcp_streams = {}

# Dict to track alerts
alert_history = {}

# List to store TCP packets
tcp_packet_list = []

# Set a timeout to ignore repeated alerts within 10s
REPEATED_ALERT_TIMEOUT = timedelta(seconds=10)


class Stream:
    """
    Used to track each TCP stream
    """
    def __init__(self, stream_id):
        self.stream_id = stream_id
        self.packets = []
        self.complete = False

    def process_packet(self, packet):
        self.packets.append(packet)

        # Check for RST or FIN flags that indicate end of stream
        # Code that did not work: if packet[TCP].flags == "R" or packet[TCP].flags == "F"
        # Check for RST (0x01) or FIN (0x04) flags
        if packet[TCP].flags & 0x01 or packet[TCP].flags & 0x04:
            self.complete = True
            print(f"[*] Finished tracking stream: {self.stream_id}")
            return True  # Stream finished
        return False

    def extract_body_and_check_malware(self):
        full_data = extract_stream_data(self.packets)
        if not full_data:
            return

        digest = hashlib.sha1(full_data).hexdigest()
        
        if digest in MALICIOUS_HASHES:
            src_ip = self.packets[0][IP].src
            dst_ip = self.packets[0][IP].dst
            alert(f"Malware detected in HTTP download! Client: {src_ip}, Server: {dst_ip}, Hash: {digest} ({MALICIOUS_HASHES[digest]})")


def get_interface():
    """
    Checks for the OS and returns the correct network interface
    """
    print("Checking for OS...")
    if platform.system() == "Windows":
        print("OS is Windows -> VMware Network Adapter VMnet1")
        return "VMware Network Adapter VMnet1"
    elif platform.system() == "Darwin":  # macOS is identified as 'Darwin'
        print("OS is Mac -> vmenet3")
        return "vmenet3"
    else:
        raise RuntimeError("Only supports windows/mac")


def alert(msg):
    """
    Alerts function to check if the alert has been logged before within the timeout
    """
    # Ignore the alert if it has already been logged before timeout
    if (msg in alert_history) and (
        alert_history[msg] + REPEATED_ALERT_TIMEOUT > datetime.now()
    ):
        return
    # Log the alert's msg and timestamp
    alert_history[msg] = datetime.now()
    print(f"*ALERT* {msg}")


def detect_malicious_ip(packet):
    """
    Detection 1.1
    Detect communication with known malicious IPs.
    """
    # Check source IP
    if packet[IP].src in MALICIOUS_IPS:
        alert(f"Malicious IP {packet[IP].src} communicating with destination: {packet[IP].dst}")
    
    # Check destination IP
    if packet[IP].dst in MALICIOUS_IPS:
        alert(f"Malicious IP {packet[IP].dst} communicating with source: {packet[IP].src}")


def detect_malicious_dns(packet):
    """
    Detection 1.2
    Detect DNS queries
    """
    # Check if packet has DNS layer
    if packet.haslayer(DNS):
        # Check if it's a DNS query (0 means query, 1 means response)
        if packet[DNS].qr == 0:
            # Get the queried name
            queried_domain = packet[DNS].qd.qname.decode('utf-8').rstrip('.')
            
            # Check if the queried domain is in our malicious domains list
            if queried_domain in MALICIOUS_DOMAINS:
                source_ip = packet[IP].src
                alert(f"DNS query for malicious domain {queried_domain} from {source_ip}")


def track_stream(packet):
    """
    Detection 1.3 Part 1
    Detect HTTP Malicious downloads via Hash
    Keep track of all streams that appears based on streamID
    """
    # Craft Stream ID in src_ip:src_port -> dst_ip:dst_port format
    streamid = f"{packet[IP].src}:{packet[TCP].sport} -> {packet[IP].dst}:{packet[TCP].dport}"

    # Check if packet's stream ID exists in the tcp_streams dictionary (previously logged streamid)
    if streamid not in tcp_streams:
        print(f"[*] Starting new stream: {streamid}")
        tcp_streams[streamid] = Stream(streamid)
        tcp_packet_list.append(tcp_streams[streamid].packets)

    stream = tcp_streams[streamid]
    is_done = stream.process_packet(packet)

    if is_done:
        stream.extract_body_and_check_malware()
        del tcp_streams[streamid]


def extract_stream_data(packets):
    """
    Detection 1.3 Part 2
    Helper function to reassemble TCP stream based on sequence number and extract the data
    """
    tcp_segments = []
    # Iterate through list of packets and append raw payload to new tcp_segments list
    for packet in packets:
        # Check packet for TCP layer and raw data (payload/download)
        if packet.haslayer(TCP) and packet.haslayer(Raw):
            payload = bytes(packet[Raw].load)
            tcp_segments.append(payload)

    # Concatenate all data segments for full payload
    stream_data = b''.join(tcp_segments)

    # Split HTTP headers and body, return just the body (raw data)
    if b'\r\n\r\n' in stream_data:
        http_body = stream_data.split(b"\r\n\r\n")[1]
        return http_body


def detect_malicious_traffic(packet):
    """
    Detect malicious network activity
    """
    try:
        # Check if the packet has IP layer
        if packet.haslayer(IP):
            detect_malicious_ip(packet)
        if packet.haslayer(DNS) and packet[DNS].qr == 0:
            detect_malicious_dns(packet)
        if packet.haslayer(TCP) and packet.sport == 80:
            track_stream(packet)

    # Print exceptions and continue
    except Exception as e:
        print(f"Error: {e}")


def main():
    print("Starting NIDS!")
    network_interface = get_interface()
    print(f"Sniffing on interface: {network_interface}")
    sniff(filter="ip", prn=detect_malicious_traffic, iface=network_interface)


if __name__ == "__main__":
    main()


Starting NIDS!
Checking for OS...
OS is Mac -> vmenet3
Sniffing on interface: vmenet3
[*] Starting new stream: 10.0.0.90:80 -> 10.0.0.7:38572
[*] Finished tracking stream: 10.0.0.90:80 -> 10.0.0.7:38572
*ALERT* Malware detected in HTTP download! Client: 10.0.0.90, Server: 10.0.0.7, Hash: 73480f2548244fb6a3e9db83a4e74082dd2fa500 (LokiBot)


### 1.4 malware signature in compressed HTTP
HTTP  responses  can  be  compressed,  which  of  course  changes  the response's hash

In [ ]:
from scapy.all import *
from datetime import datetime, timedelta
import platform
import hashlib
import gzip
import io


# Threat intelligence
MALICIOUS_IPS = ["4.2.2.2"]
MALICIOUS_DOMAINS = ["virus.com"]
MALICIOUS_HASHES = {
    "b3f067e63fff8e171ee26bcde6a6010737c8b22c": "LokiBot",
    "73480f2548244fb6a3e9db83a4e74082dd2fa500": "LokiBot",
    "efbd4555c4b881d77d28f659289373a813e79650": "TeslaCrypt", 
    "13427e27f405ea2c818d4f55745cd9fb9e336134": "TeslaCrypt"
}

# Dict to track TCP streams
tcp_streams = {}

# Dict to track alerts
alert_history = {}

# List to store TCP packets
tcp_packet_list = []

# Set a timeout to ignore repeated alerts within 10s
REPEATED_ALERT_TIMEOUT = timedelta(seconds=10)


class Stream:
    """
    Used to track each TCP stream
    """
    def __init__(self, stream_id):
        self.stream_id = stream_id
        self.packets = []
        self.complete = False

    def process_packet(self, packet):
        self.packets.append(packet)

        # Check for RST or FIN flags that indicate end of stream
        # Code that did not work: if packet[TCP].flags == "R" or packet[TCP].flags == "F"
        # Check for RST (0x01) or FIN (0x04) flags
        if packet[TCP].flags & 0x01 or packet[TCP].flags & 0x04:
            self.complete = True
            print(f"[*] Finished tracking stream: {self.stream_id}")
            return True  # Stream finished
        return False

    def extract_body_and_check_malware(self):
        full_data = extract_stream_data(self.packets)
        if not full_data:
            return

        digest = hashlib.sha1(full_data).hexdigest()
        
        if digest in MALICIOUS_HASHES:
            src_ip = self.packets[0][IP].src
            dst_ip = self.packets[0][IP].dst
            alert(f"Malware detected in HTTP download! Client: {src_ip}, Server: {dst_ip}, Hash: {digest} ({MALICIOUS_HASHES[digest]})")


def get_interface():
    """
    Checks for the OS and returns the correct network interface
    """
    print("Checking for OS...")
    if platform.system() == "Windows":
        print("OS is Windows -> VMware Network Adapter VMnet1")
        return "VMware Network Adapter VMnet1"
    elif platform.system() == "Darwin":  # macOS is identified as 'Darwin'
        print("OS is Mac -> vmenet3")
        return "vmenet3"
    else:
        raise RuntimeError("Only supports windows/mac")


def alert(msg):
    """
    Alerts function to check if the alert has been logged before within the timeout
    """
    # Ignore the alert if it has already been logged before timeout
    if (msg in alert_history) and (
        alert_history[msg] + REPEATED_ALERT_TIMEOUT > datetime.now()
    ):
        return
    # Log the alert's msg and timestamp
    alert_history[msg] = datetime.now()
    print(f"*ALERT* {msg}")


def detect_malicious_ip(packet):
    """
    Detection 1.1
    Detect communication with known malicious IPs.
    """
    # Check source IP
    if packet[IP].src in MALICIOUS_IPS:
        alert(f"Malicious IP {packet[IP].src} communicating with destination: {packet[IP].dst}")
    
    # Check destination IP
    if packet[IP].dst in MALICIOUS_IPS:
        alert(f"Malicious IP {packet[IP].dst} communicating with source: {packet[IP].src}")


def detect_malicious_dns(packet):
    """
    Detection 1.2
    Detect DNS queries
    """
    # Check if packet has DNS layer
    if packet.haslayer(DNS):
        # Check if it's a DNS query (0 means query, 1 means response)
        if packet[DNS].qr == 0:
            # Get the queried name
            queried_domain = packet[DNS].qd.qname.decode('utf-8').rstrip('.')
            
            # Check if the queried domain is in our malicious domains list
            if queried_domain in MALICIOUS_DOMAINS:
                source_ip = packet[IP].src
                alert(f"DNS query for malicious domain {queried_domain} from {source_ip}")


def track_stream(packet):
    """
    Detection 1.3 Part 1
    Detect HTTP Malicious downloads via Hash
    Keep track of all streams that appears based on streamID
    """
    # Craft Stream ID in src_ip:src_port -> dst_ip:dst_port format
    streamid = f"{packet[IP].src}:{packet[TCP].sport} -> {packet[IP].dst}:{packet[TCP].dport}"

    # Check if packet's stream ID exists in the tcp_streams dictionary (previously logged streamid)
    if streamid not in tcp_streams:
        print(f"[*] Starting new stream: {streamid}")
        tcp_streams[streamid] = Stream(streamid)
        tcp_packet_list.append(tcp_streams[streamid].packets)

    stream = tcp_streams[streamid]
    is_done = stream.process_packet(packet)

    if is_done:
        stream.extract_body_and_check_malware()
        del tcp_streams[streamid]


def extract_stream_data(packets):
    """
    Detection 1.3 Part 2
    Helper function to reassemble TCP stream based on sequence number and extract the data
    """
    tcp_segments = []
    # Iterate through list of packets and append raw payload to new tcp_segments list
    for packet in packets:
        # Check packet for TCP layer and raw data (payload/download)
        if packet.haslayer(TCP) and packet.haslayer(Raw):
            payload = bytes(packet[Raw].load)
            tcp_segments.append(payload)

    # Concatenate all data segments for full payload
    stream_data = b''.join(tcp_segments)

    # Split HTTP headers and body, return just the body (raw data)
    if b'\r\n\r\n' in stream_data:
        http_body = stream_data.split(b"\r\n\r\n")[1]
        return http_body


def extract_stream_data(packets):
    """
    Detection 1.4
    Helper function to reassemble TCP stream based on sequence number and extract the data
    Handles both uncompressed and gzip-compressed HTTP responses
    """
    tcp_segments = []
    # Iterate through list of packets and append raw payload to new tcp_segments list
    for packet in packets:
        # Check packet for TCP layer and raw data (payload/download)
        if packet.haslayer(TCP) and packet.haslayer(Raw):
            payload = bytes(packet[Raw].load)
            tcp_segments.append(payload)

    # Concatenate all data segments for full payload
    stream_data = b''.join(tcp_segments)

    # Split HTTP headers and body
    if b'\r\n\r\n' in stream_data:
        headers_part, http_body = stream_data.split(b"\r\n\r\n", 1)
        
        # Check if the response is compressed
        headers_str = headers_part.decode('utf-8', errors='ignore').lower()
        
        # Look for gzip compression in headers
        if 'content-encoding: gzip' in headers_str:
            try:
                # Decompress gzip data in memory
                decompressed_body = gzip.decompress(http_body)
                print(f"[*] Decompressed gzip data: {len(http_body)} -> {len(decompressed_body)} bytes")
                return decompressed_body
            except Exception as e:
                print(f"[!] Failed to decompress gzip data: {e}")
                return http_body
        else:
            # Return uncompressed body
            return http_body
    
    return None


def detect_malicious_traffic(packet):
    """
    Detect malicious network activity
    """
    try:
        # Check if the packet has IP layer
        if packet.haslayer(IP):
            detect_malicious_ip(packet)
        if packet.haslayer(DNS) and packet[DNS].qr == 0:
            detect_malicious_dns(packet)
        if packet.haslayer(TCP) and packet.sport == 80:
            track_stream(packet)

    # Print exceptions and continue
    except Exception as e:
        print(f"Error: {e}")


def main():
    print("Starting NIDS!")
    network_interface = get_interface()
    print(f"Sniffing on interface: {network_interface}")
    sniff(filter="ip", prn=detect_malicious_traffic, iface=network_interface)


if __name__ == "__main__":
    main()


Starting NIDS!
Checking for OS...
OS is Mac -> vmenet3
Sniffing on interface: vmenet3
[*] Starting new stream: 10.0.0.90:80 -> 10.0.0.7:41568
[*] Finished tracking stream: 10.0.0.90:80 -> 10.0.0.7:41568
[*] Decompressed gzip data: 38978 -> 38945 bytes
*ALERT* Malware detected in HTTP download! Client: 10.0.0.90, Server: 10.0.0.7, Hash: 73480f2548244fb6a3e9db83a4e74082dd2fa500 (LokiBot)


### 1.5 malware signature in FTP
dd another protocol to our repertoire of Deep-packet-inspection

In [3]:
# too hard im not doing this

### 2.1 - detect a network scan
an ARP scan - send ARPs trying to resolve all network; existing hosts will respond

In [30]:
from scapy.all import *
from datetime import datetime, timedelta
import platform
import hashlib
import gzip
import io


# Threat intelligence
MALICIOUS_IPS = ["4.2.2.2"]
MALICIOUS_DOMAINS = ["virus.com"]
MALICIOUS_HASHES = {
    "b3f067e63fff8e171ee26bcde6a6010737c8b22c": "LokiBot",
    "73480f2548244fb6a3e9db83a4e74082dd2fa500": "LokiBot",
    "efbd4555c4b881d77d28f659289373a813e79650": "TeslaCrypt", 
    "13427e27f405ea2c818d4f55745cd9fb9e336134": "TeslaCrypt"
}

# Dict to track TCP streams
tcp_streams = {}

# Dict to track alerts
alert_history = {}

# List to store TCP packets
tcp_packet_list = []

# Dict to track ARP requests and responses
arp_windows = {}

# Anything more than 20 is sus
ARP_LIMIT = 20

# Time
WINDOW_SIZE_IN_SECONDS = 20

# Set a timeout to ignore repeated alerts within 10s
REPEATED_ALERT_TIMEOUT = timedelta(seconds=10)


class Stream:
    """
    Used to track each TCP stream
    """
    def __init__(self, stream_id):
        self.stream_id = stream_id
        self.packets = []
        self.complete = False

    def process_packet(self, packet):
        self.packets.append(packet)

        # Check for RST or FIN flags that indicate end of stream
        # Code that did not work: if packet[TCP].flags == "R" or packet[TCP].flags == "F"
        # Check for RST (0x01) or FIN (0x04) flags
        if packet[TCP].flags & 0x01 or packet[TCP].flags & 0x04:
            self.complete = True
            print(f"[*] Finished tracking stream: {self.stream_id}")
            return True  # Stream finished
        return False

    def extract_body_and_check_malware(self):
        full_data = extract_stream_data(self.packets)
        if not full_data:
            return

        digest = hashlib.sha1(full_data).hexdigest()
        
        if digest in MALICIOUS_HASHES:
            src_ip = self.packets[0][IP].src
            dst_ip = self.packets[0][IP].dst
            alert(f"Malware detected in HTTP download! Client: {src_ip}, Server: {dst_ip}, Hash: {digest} ({MALICIOUS_HASHES[digest]})")


def get_interface():
    """
    Checks for the OS and returns the correct network interface
    """
    print("Checking for OS...")
    if platform.system() == "Windows":
        print("OS is Windows -> VMware Network Adapter VMnet1")
        return "VMware Network Adapter VMnet1"
    elif platform.system() == "Darwin":  # macOS is identified as 'Darwin'
        print("OS is Mac -> vmenet3")
        return "vmenet3"
    else:
        raise RuntimeError("Only supports windows/mac")


def alert(msg):
    """
    Alerts function to check if the alert has been logged before within the timeout
    """
    # Ignore the alert if it has already been logged before timeout
    if (msg in alert_history) and (
        alert_history[msg] + REPEATED_ALERT_TIMEOUT > datetime.now()
    ):
        return
    # Log the alert's msg and timestamp
    alert_history[msg] = datetime.now()
    print(f"*ALERT* {msg}")


def detect_malicious_ip(packet):
    """
    Detection 1.1
    Detect communication with known malicious IPs.
    """
    # Check source IP
    if packet[IP].src in MALICIOUS_IPS:
        alert(f"Malicious IP {packet[IP].src} communicating with destination: {packet[IP].dst}")
    
    # Check destination IP
    if packet[IP].dst in MALICIOUS_IPS:
        alert(f"Malicious IP {packet[IP].dst} communicating with source: {packet[IP].src}")


def detect_malicious_dns(packet):
    """
    Detection 1.2
    Detect DNS queries
    """
    # Check if packet has DNS layer
    if packet.haslayer(DNS):
        # Check if it's a DNS query (0 means query, 1 means response)
        if packet[DNS].qr == 0:
            # Get the queried name
            queried_domain = packet[DNS].qd.qname.decode('utf-8').rstrip('.')
            
            # Check if the queried domain is in our malicious domains list
            if queried_domain in MALICIOUS_DOMAINS:
                source_ip = packet[IP].src
                alert(f"DNS query for malicious domain {queried_domain} from {source_ip}")


def track_stream(packet):
    """
    Detection 1.3 Part 1
    Detect HTTP Malicious downloads via Hash
    Keep track of all streams that appears based on streamID
    """
    # Craft Stream ID in src_ip:src_port -> dst_ip:dst_port format
    streamid = f"{packet[IP].src}:{packet[TCP].sport} -> {packet[IP].dst}:{packet[TCP].dport}"

    # Check if packet's stream ID exists in the tcp_streams dictionary (previously logged streamid)
    if streamid not in tcp_streams:
        print(f"[*] Starting new stream: {streamid}")
        tcp_streams[streamid] = Stream(streamid)
        tcp_packet_list.append(tcp_streams[streamid].packets)

    stream = tcp_streams[streamid]
    is_done = stream.process_packet(packet)

    if is_done:
        stream.extract_body_and_check_malware()
        del tcp_streams[streamid]


def extract_stream_data(packets):
    """
    Detection 1.3 Part 2
    Helper function to reassemble TCP stream based on sequence number and extract the data
    """
    tcp_segments = []
    # Iterate through list of packets and append raw payload to new tcp_segments list
    for packet in packets:
        # Check packet for TCP layer and raw data (payload/download)
        if packet.haslayer(TCP) and packet.haslayer(Raw):
            payload = bytes(packet[Raw].load)
            tcp_segments.append(payload)

    # Concatenate all data segments for full payload
    stream_data = b''.join(tcp_segments)

    # Split HTTP headers and body, return just the body (raw data)
    if b'\r\n\r\n' in stream_data:
        http_body = stream_data.split(b"\r\n\r\n")[1]
        return http_body


def extract_stream_data(packets):
    """
    Detection 1.4
    Helper function to reassemble TCP stream based on sequence number and extract the data
    Handles both uncompressed and gzip-compressed HTTP responses
    """
    tcp_segments = []
    # Iterate through list of packets and append raw payload to new tcp_segments list
    for packet in packets:
        # Check packet for TCP layer and raw data (payload/download)
        if packet.haslayer(TCP) and packet.haslayer(Raw):
            payload = bytes(packet[Raw].load)
            tcp_segments.append(payload)

    # Concatenate all data segments for full payload
    stream_data = b''.join(tcp_segments)

    # Split HTTP headers and body
    if b'\r\n\r\n' in stream_data:
        headers_part, http_body = stream_data.split(b"\r\n\r\n", 1)
        
        # Check if the response is compressed
        headers_str = headers_part.decode('utf-8', errors='ignore').lower()
        
        # Look for gzip compression in headers
        if 'content-encoding: gzip' in headers_str:
            try:
                # Decompress gzip data in memory
                decompressed_body = gzip.decompress(http_body)
                print(f"[*] Decompressed gzip data: {len(http_body)} -> {len(decompressed_body)} bytes")
                return decompressed_body
            except Exception as e:
                print(f"[!] Failed to decompress gzip data: {e}")
                return http_body
        else:
            # Return uncompressed body
            return http_body
    
    return None


def handle_arp(packet):
    """
    Detection 2.1
    Detect a network scan (ARP)
    """
    # Get the source MAC address and the target IP
    arp_src = packet[Ether].src
    arp_target = packet[ARP].pdst
    
    # Check if there is an existing window for the source MAC, if not:
    if arp_src not in arp_windows:
        # Create a new nested dictionary using the arp_src as the key in arp_windows.
        arp_windows[arp_src] = dict()
        
        # Update the dictionary with a new key-pair value of the target IP and the timestamp of the request
        arp_windows[arp_src].update({arp_target: datetime.now()})

        # Structure of arp_windows = { <source_mac> : { <target IP> : <timestamp> } }
    else:
        # Check that the packet is not a duplicate/rebroadcast ARP request
        if arp_target not in arp_windows[arp_src]:
            # Update the dictionary with a new key-pair value of the target IP and the timestamp of the request
            arp_windows[arp_src].update({arp_target: datetime.now()})

        # Check if the current window exceeds the ARP_LIMIT value, and alert if True
        if len(arp_windows[arp_src]) > ARP_LIMIT:
            alert(f"ARP scan from {arp_src}")


def refresh_arp_window():
    """
    Detection 2.1
    Detect a network scan
    """
    # Iterate over all source MACs in arp_windows
    for arp_src in list(arp_windows.keys()):
        window_dict = arp_windows[arp_src]

        outdated_records = [
            key for key, val in window_dict.items()
            if (val + timedelta(seconds=WINDOW_SIZE_IN_SECONDS) < datetime.now())
        ]
        for key in outdated_records:
            del window_dict[key]

        # Free up memory
        if not window_dict:
            del arp_windows[arp_src]


def detect_malicious_traffic(packet):
    """
    Detect malicious network activity
    """
    try:
        # Check if the packet has IP layer
        if packet.haslayer(IP):
            detect_malicious_ip(packet)
        if packet.haslayer(DNS) and packet[DNS].qr == 0:
            detect_malicious_dns(packet)
        if packet.haslayer(TCP) and packet.sport == 80:
            track_stream(packet)
        if packet.haslayer(ARP):
            refresh_arp_window()
            handle_arp(packet)

    # Print exceptions and continue
    except Exception as e:
        print(f"Error: {e}")


def main():
    print("Starting NIDS!")
    network_interface = get_interface()
    print(f"Sniffing on interface: {network_interface}")
    sniff(filter="ip or arp", prn=detect_malicious_traffic, iface=network_interface)


if __name__ == "__main__":
    main()


Starting NIDS!
Checking for OS...
OS is Mac -> vmenet3
Sniffing on interface: vmenet3
*ALERT* ARP scan from 02:42:0a:00:00:0d


### 2.2 - detect a TCP scan
attackers can use a TCP scan / service enumeration to discover open ports on a server

In [1]:
from scapy.all import *
from datetime import datetime, timedelta
import platform
import hashlib
import gzip
import io


# Threat intelligence
MALICIOUS_IPS = ["4.2.2.2"]
MALICIOUS_DOMAINS = ["virus.com"]
MALICIOUS_HASHES = {
    "b3f067e63fff8e171ee26bcde6a6010737c8b22c": "LokiBot",
    "73480f2548244fb6a3e9db83a4e74082dd2fa500": "LokiBot",
    "efbd4555c4b881d77d28f659289373a813e79650": "TeslaCrypt", 
    "13427e27f405ea2c818d4f55745cd9fb9e336134": "TeslaCrypt"
}

# Dict to track TCP streams
tcp_streams = {}

# Dict to track alerts
alert_history = {}

# List to store TCP packets
tcp_packet_list = []

# Dict to track ARP requests and responses
arp_windows = {}

# Dict to track TCP windows
tcp_windows = {}

# Anything more than 20 is sus
ARP_LIMIT = 20

# Anything more than 20 is sus
SYN_LIMIT = 20

# Time
WINDOW_SIZE_IN_SECONDS = 20

# Set a timeout to ignore repeated alerts within 10s
REPEATED_ALERT_TIMEOUT = timedelta(seconds=10)


class Stream:
    """
    Used to track each TCP stream
    """
    def __init__(self, stream_id):
        self.stream_id = stream_id
        self.packets = []
        self.complete = False

    def process_packet(self, packet):
        self.packets.append(packet)

        # Check for RST or FIN flags that indicate end of stream
        # Code that did not work: if packet[TCP].flags == "R" or packet[TCP].flags == "F"
        # Check for RST (0x01) or FIN (0x04) flags
        if packet[TCP].flags & 0x01 or packet[TCP].flags & 0x04:
            self.complete = True
            print(f"[*] Finished tracking stream: {self.stream_id}")
            return True  # Stream finished
        return False

    def extract_body_and_check_malware(self):
        full_data = extract_stream_data(self.packets)
        if not full_data:
            return

        digest = hashlib.sha1(full_data).hexdigest()
        
        if digest in MALICIOUS_HASHES:
            src_ip = self.packets[0][IP].src
            dst_ip = self.packets[0][IP].dst
            alert(f"Malware detected in HTTP download! Client: {src_ip}, Server: {dst_ip}, Hash: {digest} ({MALICIOUS_HASHES[digest]})")


def get_interface():
    """
    Checks for the OS and returns the correct network interface
    """
    print("Checking for OS...")
    if platform.system() == "Windows":
        print("OS is Windows -> VMware Network Adapter VMnet1")
        return "VMware Network Adapter VMnet1"
    elif platform.system() == "Darwin":  # macOS is identified as 'Darwin'
        print("OS is Mac -> vmenet3")
        return "vmenet3"
    else:
        raise RuntimeError("Only supports windows/mac")


def alert(msg):
    """
    Alerts function to check if the alert has been logged before within the timeout
    """
    # Ignore the alert if it has already been logged before timeout
    if (msg in alert_history) and (
        alert_history[msg] + REPEATED_ALERT_TIMEOUT > datetime.now()
    ):
        return
    # Log the alert's msg and timestamp
    alert_history[msg] = datetime.now()
    print(f"*ALERT* {msg}")


def detect_malicious_ip(packet):
    """
    Detection 1.1
    Detect communication with known malicious IPs.
    """
    # Check source IP
    if packet[IP].src in MALICIOUS_IPS:
        alert(f"Malicious IP {packet[IP].src} communicating with destination: {packet[IP].dst}")
    
    # Check destination IP
    if packet[IP].dst in MALICIOUS_IPS:
        alert(f"Malicious IP {packet[IP].dst} communicating with source: {packet[IP].src}")


def detect_malicious_dns(packet):
    """
    Detection 1.2
    Detect DNS queries
    """
    # Check if packet has DNS layer
    if packet.haslayer(DNS):
        # Check if it's a DNS query (0 means query, 1 means response)
        if packet[DNS].qr == 0:
            # Get the queried name
            queried_domain = packet[DNS].qd.qname.decode('utf-8').rstrip('.')
            
            # Check if the queried domain is in our malicious domains list
            if queried_domain in MALICIOUS_DOMAINS:
                source_ip = packet[IP].src
                alert(f"DNS query for malicious domain {queried_domain} from {source_ip}")


def track_stream(packet):
    """
    Detection 1.3 Part 1
    Detect HTTP Malicious downloads via Hash
    Keep track of all streams that appears based on streamID
    """
    # Craft Stream ID in src_ip:src_port -> dst_ip:dst_port format
    streamid = f"{packet[IP].src}:{packet[TCP].sport} -> {packet[IP].dst}:{packet[TCP].dport}"

    # Check if packet's stream ID exists in the tcp_streams dictionary (previously logged streamid)
    if streamid not in tcp_streams:
        print(f"[*] Starting new stream: {streamid}")
        tcp_streams[streamid] = Stream(streamid)
        tcp_packet_list.append(tcp_streams[streamid].packets)

    stream = tcp_streams[streamid]
    is_done = stream.process_packet(packet)

    if is_done:
        stream.extract_body_and_check_malware()
        del tcp_streams[streamid]


def extract_stream_data(packets):
    """
    Detection 1.3 Part 2
    Helper function to reassemble TCP stream based on sequence number and extract the data
    """
    tcp_segments = []
    # Iterate through list of packets and append raw payload to new tcp_segments list
    for packet in packets:
        # Check packet for TCP layer and raw data (payload/download)
        if packet.haslayer(TCP) and packet.haslayer(Raw):
            payload = bytes(packet[Raw].load)
            tcp_segments.append(payload)

    # Concatenate all data segments for full payload
    stream_data = b''.join(tcp_segments)

    # Split HTTP headers and body, return just the body (raw data)
    if b'\r\n\r\n' in stream_data:
        http_body = stream_data.split(b"\r\n\r\n")[1]
        return http_body


def extract_stream_data(packets):
    """
    Detection 1.4
    Helper function to reassemble TCP stream based on sequence number and extract the data
    Handles both uncompressed and gzip-compressed HTTP responses
    """
    tcp_segments = []
    # Iterate through list of packets and append raw payload to new tcp_segments list
    for packet in packets:
        # Check packet for TCP layer and raw data (payload/download)
        if packet.haslayer(TCP) and packet.haslayer(Raw):
            payload = bytes(packet[Raw].load)
            tcp_segments.append(payload)

    # Concatenate all data segments for full payload
    stream_data = b''.join(tcp_segments)

    # Split HTTP headers and body
    if b'\r\n\r\n' in stream_data:
        headers_part, http_body = stream_data.split(b"\r\n\r\n", 1)
        
        # Check if the response is compressed
        headers_str = headers_part.decode('utf-8', errors='ignore').lower()
        
        # Look for gzip compression in headers
        if 'content-encoding: gzip' in headers_str:
            try:
                # Decompress gzip data in memory
                decompressed_body = gzip.decompress(http_body)
                print(f"[*] Decompressed gzip data: {len(http_body)} -> {len(decompressed_body)} bytes")
                return decompressed_body
            except Exception as e:
                print(f"[!] Failed to decompress gzip data: {e}")
                return http_body
        else:
            # Return uncompressed body
            return http_body
    
    return None


def handle_arp(packet):
    """
    Detection 2.1
    Detect a network scan (ARP)
    """
    # Get the source MAC address and the target IP
    arp_src = packet[Ether].src
    arp_target = packet[ARP].pdst
    
    # Check if there is an existing window for the source MAC, if not:
    if arp_src not in arp_windows:
        # Create a new nested dictionary using the arp_src as the key in arp_windows.
        arp_windows[arp_src] = dict()
        
        # Update the dictionary with a new key-pair value of the target IP and the timestamp of the request
        arp_windows[arp_src].update({arp_target: datetime.now()})

        # Structure of arp_windows = { <source_mac> : { <target IP> : <timestamp> } }
    else:
        # Check that the packet is not a duplicate/rebroadcast ARP request
        if arp_target not in arp_windows[arp_src]:
            # Update the dictionary with a new key-pair value of the target IP and the timestamp of the request
            arp_windows[arp_src].update({arp_target: datetime.now()})

        # Check if the current window exceeds the ARP_LIMIT value, and alert if True
        if len(arp_windows[arp_src]) > ARP_LIMIT:
            alert(f"ARP scan from {arp_src}")


def refresh_windows(target_windows):
    """
    Detection 2.1
    Detect a network scan
    """
    # Iterate over all source MACs in arp_windows
    for src in list(target_windows.keys()):
        window_dict = target_windows[src]

        outdated_records = [
            key for key, val in window_dict.items()
            if (val + timedelta(seconds=WINDOW_SIZE_IN_SECONDS) < datetime.now())
        ]
        for key in outdated_records:
            del window_dict[key]

        # Free up memory
        if not window_dict:
            del target_windows[src]


def handle_tcp_syn(packet):
    """
    Detection 2.2
    Detect a TCP scan
    """
    # Get the source IP address, target IP address and destination port
    ip_src = packet[IP].src
    ip_dst = packet[IP].dst
    port_dst = packet[TCP].dport
    
    # Create an identifier for source ip -> destination ip
    syn_identifier = f"{ip_src} -> {ip_dst}"

    # Check if there is an existing window for the identifier, if not:
    if syn_identifier not in tcp_windows:
        # Create a new nested dictionary using the syn_identifier as the key in tcp_windows.
        tcp_windows[syn_identifier] = dict()
        
        # Update the dictionary with a new key-pair value of the destination port and the timestamp of the request
        tcp_windows[syn_identifier].update({port_dst: datetime.now()})
        
        # Structure of tcp_windows = { <syn_identifier> : { <destination_port> : <timestamp> } }
    else:
        # Check that the packet is not a duplicate/rebroadcast TCP SYN
        if port_dst not in tcp_windows[syn_identifier]:
            # Update the dictionary with a new key-pair value of the destination port and the timestamp of the request
            tcp_windows[syn_identifier].update({port_dst: datetime.now()})
           
        # Check if the current window exceeds the SYN_LIMIT value, and alert if True
        if len(tcp_windows[syn_identifier]) > SYN_LIMIT:
            alert(f"TCP SYN Scan (Port Scan) from {ip_src} tp {ip_dst}")


def detect_malicious_traffic(packet):
    """
    Detect malicious network activity
    """
    try:
        # Check if the packet has IP layer
        if packet.haslayer(IP):
            detect_malicious_ip(packet)
        if packet.haslayer(DNS) and packet[DNS].qr == 0:
            detect_malicious_dns(packet)
        if packet.haslayer(TCP) and packet.sport == 80:
            track_stream(packet)
        if packet.haslayer(ARP):
            refresh_windows(arp_windows)
            handle_arp(packet)
        if packet.haslayer(TCP) and 'S' in packet[TCP].flags:
            refresh_windows(tcp_windows)
            handle_tcp_syn(packet)

    # Print exceptions and continue
    except Exception as e:
        print(f"Error: {e}")


def main():
    print("Starting NIDS!")
    network_interface = get_interface()
    print(f"Sniffing on interface: {network_interface}")
    sniff(filter="ip or arp", prn=detect_malicious_traffic, iface=network_interface)


if __name__ == "__main__":
    main()


Starting NIDS!
Checking for OS...
OS is Mac -> vmenet3
Sniffing on interface: vmenet3
*ALERT* Malicious IP 4.2.2.2 communicating with source: 10.0.0.13
*ALERT* Malicious IP 4.2.2.2 communicating with destination: 10.0.0.13


### misc for endpoint detection

In [ ]:
import os
import time

CALC_NAME = "CalculatorApp.exe"

def check_for_calc():
    # Get the list of all running processes
    processes = os.popen('tasklist').read()
    
    if CALC_NAME in processes:
        print(f"[*] {CALC_NAME} is running!")
    else:
        print(f"[*] {CALC_NAME} is not running.")

def main():
    print("Waiting for {CALC_NAME}")
    while True:
        check_for_calc()
        time.sleep(1)

if __name__ == "__main__":
    main()